In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
from tqdm import tqdm    # Shows progress bar
import torch.optim as optim
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
import kagglehub
import os
import random
import numpy as np
import torchdata
import torch.nn as nn

import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch.optim as optim

from sklearn.metrics import confusion_matrix
import seaborn as sns
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from torchvision import models
import torch.nn as nn
import torch



In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader


class SubtractOne(nn.Module):
  def forward(self, img):
    return img-1


# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),# TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),                                                      # Convert to tensor Convert to Tensor
    #SubtractOne(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
])


# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'



# Create DataLoaders and display samples
# Write your code here
# Each batch contains 32 images.
batch_size = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,                       # Groups data into batches of BS
    shuffle=True,                                # for Training (prevents learning order patterns)
    num_workers=2
    )


valid_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,                                # No shuffle on Test
    num_workers=2
    )


# Check a batch of images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, \nLabels: {labels}")
print('\n\n')



# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])


# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].axis('off')

plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
#  Choose GPU if available; otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Load pretrained EfficientNetV2-S model
#efficientnet = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1) # or pretrained=True both works

efficientnet = models.efficientnet_v2_s(pretrained=True ) # or pretrained=True both works


# Freeze the backbone (feature extractor) - we only want to train the classifier head

efficientnet.requires_grad_(False)



# Modify the classifier
efficientnet.classifier[1] = nn.Linear(                                                # Creates a FC layer: "Take in_features, Produce out_features”
    efficientnet.classifier[1].in_features,                                            # in_features = length = 1280
    26                                                                           #  out_features = one value only D/C = 1
    )



# Set to Eval Mode  +  Move to GPU if available                                 ; .eval(): turns off dropout and makes BatchNorm use stored running stats  +   .to(device): moves model parameters to GPU/CPU
efficientnet.eval().to(device)


In [ ]:
# Check if in/out_features correct
print(efficientnet.classifier[1])


In [ ]:
# Write your code here
# Training Loop

# Training Loop

def train_one_epoch(model, dataloader, criterion, optimizer, device):

    # Set Model to Training Mode                                                ; Enables layers like: Dropout, Batch Normalization (Training behavior)
    model.train()

    # initilize
    running_loss = 0.0                                                          # sum of losses over all samples (we’ll average later)
    correct_predictions = 0                                                     # how many predictions were correct
    total_samples = 0                                                           # Total Num of samples


    # Loop Over Batches
    for images, labels in tqdm(dataloader):                                     # [B, 3, 32, 32]

        # Move data to GPU if available                                         ; Moves data to the SAME device as the model.
        images, labels = images.to(device), labels.to(device)


        # Reset old Gradients
        optimizer.zero_grad()


        # ----------------------------------------------------------Forward pass
        outputs = model(images)

        # Compute loss: softmax + negative log likelihood - Outputs [B]
        loss = criterion(outputs, labels)



        # -------------------------------------------------------Backpropagation


        # Backpropagation (compute Gradients)
        loss.backward()

        # Update model parameters using computed Gradients
        optimizer.step()


        # ------------------------------------------------------------Track Loss
        # Collect the loss
        total_loss += loss.item()


        # --------------------------------------------------------Track Accuracy
        # Get predicted class
        # Collect the loss
        running_loss += loss.item() * images.size(0)

        # Get predicted class - shape: [B] containing class indices
        _, predicted = torch.max(outputs.data, 1)

        # Update counters
        total_samples += labels.size(0)                                         # Add batch size to total samples
        correct_predictions += (predicted == labels).sum().item()               # Count how many predictions match true labels



    # Calc Average loss & accuracy for the whole epoch - Compute epoch metrics
    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy







In [ ]:
# Validation Loop

def validate(model, dataloader, criterion, device):

    # Set Model to Evaluation Mode                                              ;  Dropout → disabled  -   BatchNorm → uses stored statistics
    model.eval()

    total_loss = 0                                                              # Accumulate validation loss
    correct = 0                                                                 # Count correct predictions
    total = 0                                                                   # Total Num of samples


    # Disable Gradient calculation                                              ; No Backpropagation in Validation
    with torch.no_grad():

        # Loop Over Val Batches
        for images, labels in dataloader:

            # Move data to GPU if available                                     ; Moves data to the SAME device as the model.
            images, labels = images.to(device), labels.to(device)


            # ------------------------------------------------------Forward pass
            # Model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
            outputs = model(images).squeeze()

            # Compute loss
            loss = criterion(outputs, labels)

            # --------------------------------------------------------Track Loss
            # Collect the loss
            total_loss += loss.item()


            # ------------------------------------------------- Compute Accuracy
            # Get predicted class
            predictions = torch.sigmoid(outputs) > 0.5

            # Compares predictions with true labels  -  Counts how many are correct
            correct += (predictions == labels).sum().item()

            # Adds Num of samples in the current batch
            total += labels.size(0)


    # Calc average validation loss
    avg_loss = total_loss / len(dataloader)

    # Calc accuracy in percentage
    accuracy = 100 * correct / total


    # Returns validation loss & accuracy
    return avg_loss, accuracy


In [ ]:
print(train_loader)

In [ ]:
# Write your code here
# Training setup

# Define loss criterion
criterion = nn.CrossEntropyLoss()


# Set learning rate
learning_rate = 0.001

# Set number of epochs
num_epochs = 10


# Define optimizer
#Pass model.parameters() and learning_rate
optimizer = optim.Adam(efficientnet.parameters(), lr= learning_rate)




# Initialize history tracking
history = {"train_loss":[], "train_acc": [], "test_loss": [], "test_acc": []}



# Training Process:                                                             ; repeat Training + Validation for each Epoch
print("Starting Training...")
print('*'*70)

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        efficientnet,
        train_loader,
        criterion,
        optimizer,
        device
        )

    test_loss, test_acc = validate(
        efficientnet,
        valid_loader,
        criterion
        )

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
# Write your code here
